In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import string
from collections import Counter
import pickle
import warnings
warnings.filterwarnings('ignore')

class IndonesianTextPreprocessor:
    """Preprocessing khusus untuk teks bahasa Indonesia"""
    
    def __init__(self):
        # Stopwords bahasa Indonesia yang umum
        self.stopwords = {
            'dan', 'atau', 'yang', 'di', 'ke', 'dari', 'pada', 'dengan', 'untuk', 
            'dalam', 'adalah', 'akan', 'ada', 'juga', 'tidak', 'ini', 'itu', 
            'atau', 'saja', 'hanya', 'bisa', 'dapat', 'sudah', 'telah', 'sedang',
            'kemudian', 'lalu', 'setelah', 'sambil', 'hingga', 'sampai', 'agar',
            'supaya', 'karena', 'sebab', 'oleh', 'secara', 'seperti', 'ibarat'
        }
        
        # Kata-kata umum dalam resep yang bisa dihapus
        self.recipe_stopwords = {
            'buah', 'lembar', 'siung', 'butir', 'biji', 'potong', 'iris', 'cincang',
            'parut', 'haluskan', 'aduk', 'masukkan', 'tambahkan', 'tuang', 'panaskan',
            'goreng', 'rebus', 'kukus', 'bakar', 'panggang', 'secukupnya', 'seperlunya','aduk2'
        }
        
    def clean_text(self, text):
        """Membersihkan teks bahasa Indonesia"""
        if pd.isna(text) or text == '':
            return ''
            
        # Convert to lowercase
        text = str(text).lower()
        
        # Remove numbers and punctuation
        text = re.sub(r'\d+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        
        # Remove stopwords
        words = text.split()
        words = [word for word in words if word not in self.stopwords and word not in self.recipe_stopwords]
        
        return ' '.join(words)
    
    def extract_keywords(self, text, max_words=10):
        """Ekstrak kata kunci penting dari teks"""
        cleaned = self.clean_text(text)
        words = cleaned.split()
        
        # Ambil kata-kata yang paling sering muncul
        word_freq = Counter(words)
        return [word for word, freq in word_freq.most_common(max_words)]

class EnhancedIndonesianRecipeRecommender:
    """Enhanced Sistem rekomendasi resep makanan Indonesia dengan output lengkap"""
    
    def __init__(self):
        self.text_processor = IndonesianTextPreprocessor()
        self.model = None
        self.encoders = {}
        self.scalers = {}
        self.tfidf_vectorizer = None
        self.processed_data = None
        self.original_data = None  # Simpan data asli untuk output lengkap
        self.category_similarity = None
        
    def preprocess_data(self, df):
        """Preprocessing data resep Indonesia"""
        print("🔄 Memulai preprocessing data...")
        
        # Simpan data asli
        self.original_data = df.copy()
        
        # Copy dataframe
        data = df.copy()
        
        # 1. Preprocessing teks berbahasa Indonesia
        print("📝 Preprocessing teks bahasa Indonesia...")
        data['Category_Cleaned'] = data['Category'].apply(self.text_processor.clean_text)
        data['Title_Keywords'] = data['Title Cleaned'].apply(
            lambda x: ' '.join(self.text_processor.extract_keywords(x, 5))
        )
        data['Ingredients_Keywords'] = data['Ingredients Cleaned'].apply(
            lambda x: ' '.join(self.text_processor.extract_keywords(x, 10))
        )
        data['Steps_Keywords'] = data['Steps Cleaned'].apply(
            lambda x: ' '.join(self.text_processor.extract_keywords(x, 8))
        )
        
        # 2. Feature engineering untuk difficulty
        print("⚙️ Menghitung tingkat kesulitan resep...")
        
        # Hitung difficulty score berdasarkan ingredients dan steps
        data['Ingredient_Complexity'] = data['Total Ingredients'].apply(
            lambda x: 1 if x <= 5 else 2 if x <= 10 else 3
        )
        data['Steps_Complexity'] = data['Total Steps'].apply(
            lambda x: 1 if x <= 5 else 2 if x <= 10 else 3
        )
        data['Difficulty_Score'] = (data['Ingredient_Complexity'] + data['Steps_Complexity']) / 2
        
        # 3. Normalisasi rating
        data['Rating_Normalized'] = data['rating'] / 5.0
        
        # Proper handling of total_rating normalization
        total_rating_scaler = MinMaxScaler()
        data['Total_Rating_Normalized'] = total_rating_scaler.fit_transform(
            data[['total_rating']].fillna(data['total_rating'].mean())
        ).flatten()
        self.scalers['total_rating'] = total_rating_scaler
        
        # 4. Encoding kategori
        print("🏷️ Encoding kategori...")
        category_encoder = LabelEncoder()
        data['Category_Encoded'] = category_encoder.fit_transform(data['Category_Cleaned'])
        self.encoders['category'] = category_encoder
        
        # 5. User dan Item encoding
        user_encoder = LabelEncoder()
        item_encoder = LabelEncoder()
        data['User_Encoded'] = user_encoder.fit_transform(data['user_id'])
        data['Item_Encoded'] = item_encoder.fit_transform(data['item_id'])
        self.encoders['user'] = user_encoder
        self.encoders['item'] = item_encoder
        
        # 6. TF-IDF untuk content similarity
        print("📊 Membuat TF-IDF vectors...")
        combined_text = (
            data['Title_Keywords'] + ' ' + 
            data['Ingredients_Keywords'] + ' ' + 
            data['Category_Cleaned']
        )
        
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=1000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2)
        )
        tfidf_matrix = self.tfidf_vectorizer.fit_transform(combined_text)
        
        # Simpan similarity matrix
        self.category_similarity = cosine_similarity(tfidf_matrix)
        
        # 7. Scaling numerical features
        scaler = StandardScaler()
        numerical_features = ['Total Ingredients', 'Total Steps', 'Difficulty_Score']
        data[numerical_features] = scaler.fit_transform(data[numerical_features])
        self.scalers['numerical'] = scaler
        
        self.processed_data = data
        print("✅ Preprocessing selesai!")
        
        return data
    
    def build_model(self, n_users, n_items, n_categories, embedding_dim=50):
        """Membangun model hybrid recommendation"""
        print("🏗️ Membangun model rekomendasi hybrid...")
        
        # Input layers
        user_input = Input(shape=(), name='user_id')
        item_input = Input(shape=(), name='item_id')
        category_input = Input(shape=(), name='category')
        
        # Numerical features input
        numerical_input = Input(shape=(3,), name='numerical_features')
        
        # Embedding layers
        user_embedding = Embedding(n_users, embedding_dim, name='user_embedding')(user_input)
        item_embedding = Embedding(n_items, embedding_dim, name='item_embedding')(item_input)
        category_embedding = Embedding(n_categories, embedding_dim//2, name='category_embedding')(category_input)
        
        # Flatten embeddings
        user_vec = tf.keras.layers.Flatten()(user_embedding)
        item_vec = tf.keras.layers.Flatten()(item_embedding)
        category_vec = tf.keras.layers.Flatten()(category_embedding)
        
        # Concatenate all features
        concat = Concatenate()([user_vec, item_vec, category_vec, numerical_input])
        
        # Deep layers
        x = Dense(256, activation='relu')(concat)
        x = BatchNormalization()(x)
        x = Dropout(0.3)(x)
        
        x = Dense(128, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)
        
        x = Dense(64, activation='relu')(x)
        x = Dropout(0.1)(x)
        
        # Output layer
        output = Dense(1, activation='sigmoid', name='rating')(x)
        
        # Create model
        model = Model(
            inputs=[user_input, item_input, category_input, numerical_input],
            outputs=output
        )
        
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='mse',
            metrics=['mae']
        )
        
        self.model = model
        return model
    
    def prepare_training_data(self, data):
        """Menyiapkan data untuk training"""
        X = {
            'user_id': data['User_Encoded'].values,
            'item_id': data['Item_Encoded'].values,
            'category': data['Category_Encoded'].values,
            'numerical_features': data[['Total Ingredients', 'Total Steps', 'Difficulty_Score']].values
        }
        y = data['Rating_Normalized'].values
        
        return X, y
    
    def train_model(self, data, test_size=0.2, validation_size=0.1, epochs=100, batch_size=512):
        """Training model dengan validasi"""
        print("🚀 Memulai training model...")
        
        # Prepare data
        X, y = self.prepare_training_data(data)
        
        # Get indices for splitting
        indices = np.arange(len(y))
        
        # First split: train+val vs test
        train_val_indices, test_indices = train_test_split(
            indices, 
            test_size=test_size, 
            random_state=42, 
            stratify=data['Category_Encoded']
        )
        
        # Second split: train vs val
        train_indices, val_indices = train_test_split(
            train_val_indices,
            test_size=validation_size/(1-test_size), 
            random_state=42,
            stratify=data['Category_Encoded'].iloc[train_val_indices]
        )
        
        # Create splits using indices
        X_train = {
            'user_id': X['user_id'][train_indices],
            'item_id': X['item_id'][train_indices],
            'category': X['category'][train_indices],
            'numerical_features': X['numerical_features'][train_indices]
        }
        
        X_val = {
            'user_id': X['user_id'][val_indices],
            'item_id': X['item_id'][val_indices],
            'category': X['category'][val_indices],
            'numerical_features': X['numerical_features'][val_indices]
        }
        
        X_test = {
            'user_id': X['user_id'][test_indices],
            'item_id': X['item_id'][test_indices],
            'category': X['category'][test_indices],
            'numerical_features': X['numerical_features'][test_indices]
        }
        
        y_train = y[train_indices]
        y_val = y[val_indices]
        y_test = y[test_indices]
        
        # Build model
        n_users = len(self.encoders['user'].classes_)
        n_items = len(self.encoders['item'].classes_)
        n_categories = len(self.encoders['category'].classes_)
        
        self.build_model(n_users, n_items, n_categories)
        
        # Callbacks
        callbacks = [
            EarlyStopping(patience=10, restore_best_weights=True),
            ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6)
        ]
        
        # Train model
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        # Evaluate
        train_loss, train_mae = self.model.evaluate(X_train, y_train, verbose=0)
        val_loss, val_mae = self.model.evaluate(X_val, y_val, verbose=0)
        test_loss, test_mae = self.model.evaluate(X_test, y_test, verbose=0)
        
        return {
            'history': history,
            'train_metrics': {'rmse': np.sqrt(train_loss), 'mae': train_mae},
            'val_metrics': {'rmse': np.sqrt(val_loss), 'mae': val_mae},
            'test_metrics': {'rmse': np.sqrt(test_loss), 'mae': test_mae},
            'test_data': (X_test, y_test),
            'test_indices': test_indices
        }
    
    def get_enhanced_recommendations(self, user_id, top_k=10, category_filter=None, difficulty_max=3, 
                                  min_rating=3.0, show_detailed=True):
        """Mendapatkan rekomendasi dengan output lengkap seperti Netflix"""
        if self.model is None:
            raise ValueError("Model belum ditraining!")
        
        print(f"\n🎯 Mencari rekomendasi untuk User ID: {user_id}")
        print("=" * 60)
        
        # Encode user_id
        try:
            user_encoded = self.encoders['user'].transform([user_id])[0]
            user_type = "existing"
        except:
            # User baru, gunakan pendekatan content-based
            print("👤 User baru terdeteksi - menggunakan content-based recommendations")
            user_type = "new"
            return self._get_enhanced_content_based_recommendations(
                category_filter, difficulty_max, top_k, min_rating, show_detailed
            )
        
        # Get all items yang belum di-rate user ini
        user_data = self.processed_data[self.processed_data['user_id'] == user_id]
        rated_items = set(user_data['item_id'].values)
        all_items = set(self.processed_data['item_id'].values)
        unrated_items = list(all_items - rated_items)
        
        print(f"📊 User telah menilai {len(rated_items)} resep dari total {len(all_items)} resep")
        
        if not unrated_items:
            print("⚠️  User sudah menilai semua resep!")
            return []
        
        # Prepare prediction data
        n_items = len(unrated_items)
        pred_data = {
            'user_id': np.full(n_items, user_encoded),
            'item_id': self.encoders['item'].transform(unrated_items),
            'category': [],
            'numerical_features': []
        }
        
        # Get item features
        for item_id in unrated_items:
            item_data = self.processed_data[self.processed_data['item_id'] == item_id].iloc[0]
            pred_data['category'].append(item_data['Category_Encoded'])
            pred_data['numerical_features'].append([
                item_data['Total Ingredients'],
                item_data['Total Steps'], 
                item_data['Difficulty_Score']
            ])
        
        pred_data['category'] = np.array(pred_data['category'])
        pred_data['numerical_features'] = np.array(pred_data['numerical_features'])
        
        # Predict ratings
        predictions = self.model.predict(pred_data, verbose=0).flatten()
        
        # Create recommendation list dengan info lengkap
        recommendations = []
        for i, item_id in enumerate(unrated_items):
            # Ambil data dari processed data
            item_data = self.processed_data[self.processed_data['item_id'] == item_id].iloc[0]
            
            # Ambil data asli untuk output lengkap
            original_item_data = self.original_data[self.original_data['item_id'] == item_id].iloc[0]
            
            predicted_rating = predictions[i] * 5  # Scale back to 1-5
            
            # Apply filters
            if category_filter and item_data['Category'] != category_filter:
                continue
            if item_data['Difficulty_Score'] > difficulty_max:
                continue
            if predicted_rating < min_rating:
                continue
                
            # Hitung difficulty level yang lebih readable
            if item_data['Difficulty_Score'] <= 1.5:
                difficulty_level = "Mudah"
            elif item_data['Difficulty_Score'] <= 2.5:
                difficulty_level = "Sedang"
            else:
                difficulty_level = "Sulit"
                
            recommendations.append({
                'item_id': item_id,
                'title_cleaned': original_item_data['Title Cleaned'],
                'steps_cleaned': original_item_data['Steps Cleaned'],
                'ingredients_cleaned': original_item_data['Ingredients Cleaned'],
                'category': original_item_data['Category'],
                'total_rating': original_item_data['total_rating'],
                'image_url': original_item_data.get('Image URL', 'N/A'),
                'predicted_rating': predicted_rating,
                'difficulty_level': difficulty_level,
                'difficulty_score': item_data['Difficulty_Score'],
                'total_ingredients': original_item_data['Total Ingredients'],
                'total_steps': original_item_data['Total Steps'],
                'user_type': user_type
            })
        
        # Sort by predicted rating
        recommendations.sort(key=lambda x: x['predicted_rating'], reverse=True)
        final_recommendations = recommendations[:top_k]
        
        # Display recommendations
        if show_detailed:
            self._display_recommendations(user_id, final_recommendations, user_type)
        
        return final_recommendations
    
    def _get_enhanced_content_based_recommendations(self, category_filter=None, difficulty_max=3, 
                                                  top_k=10, min_rating=3.0, show_detailed=True):
        """Rekomendasi berbasis konten untuk user baru dengan output lengkap"""
        data = self.processed_data.copy()
        original_data = self.original_data.copy()
        
        # Apply filters
        if category_filter:
            mask = data['Category'] == category_filter
            data = data[mask]
            original_data = original_data[mask]
            
        data = data[data['Difficulty_Score'] <= difficulty_max]
        data = data[data['total_rating'] >= min_rating]
        
        # Get corresponding original data
        item_ids = data['item_id'].tolist()
        original_data = original_data[original_data['item_id'].isin(item_ids)]
        
        # Sort by total_rating and popularity
        merged_data = data.merge(original_data[['item_id', 'Title Cleaned', 'Steps Cleaned', 
                                             'Ingredients Cleaned', 'Image URL']], on='item_id')
        merged_data = merged_data.sort_values(['total_rating', 'rating'], ascending=False)
        
        recommendations = []
        for _, row in merged_data.head(top_k).iterrows():
            # Hitung difficulty level yang lebih readable
            if row['Difficulty_Score'] <= 1.5:
                difficulty_level = "Cepat & Mudah"
            elif row['Difficulty_Score'] <= 2.5:
                difficulty_level = "Butuh Usaha"
            else:
                difficulty_level = "Level Dewa Masak"
                
            recommendations.append({
                'item_id': row['item_id'],
                'title_cleaned': row['Title Cleaned'],
                'steps_cleaned': row['Steps Cleaned'],
                'ingredients_cleaned': row['Ingredients Cleaned'],
                'category': row['Category'],
                'total_rating': row['total_rating'],
                'image_url': row.get('Image URL', 'N/A'),
                'predicted_rating': row['total_rating'],  # Untuk user baru, gunakan actual rating
                'difficulty_level': difficulty_level,
                'difficulty_score': row['Difficulty_Score'],
                'total_ingredients': row['Total Ingredients'],
                'total_steps': row['Total Steps'],
                'user_type': 'new'
            })
        
        if show_detailed:
            self._display_recommendations("NEW_USER", recommendations, "new")
            
        return recommendations
    
    def _display_recommendations(self, user_id, recommendations, user_type):
        """Display recommendations dalam format seperti Netflix"""
        print(f"\n🍽️  REKOMENDASI RESEP UNTUK USER: {user_id}")
        print(f"👤 Tipe User: {'Existing User (Collaborative Filtering)' if user_type == 'existing' else 'New User (Content-Based)'}")
        print("=" * 80)
        
        if not recommendations:
            print("❌ Tidak ada rekomendasi yang memenuhi kriteria")
            return
        
        for i, rec in enumerate(recommendations, 1):
            print(f"\n🥘 REKOMENDASI #{i}")
            print("-" * 50)
            print(f"📍 Item ID: {rec['item_id']}")
            print(f"🍲 Judul: {rec['title_cleaned']}")
            print(f"🏷️  Kategori: {rec['category']}")
            print(f"⭐ Predicted Rating: {rec['predicted_rating']:.2f}/5.0")
            print(f"👥 Total Rating: {rec['total_rating']}")
            print(f"📊 Tingkat Kesulitan: {rec['difficulty_level']} ({rec['difficulty_score']:.2f})")
            print(f"🧾 Total Bahan: {rec['total_ingredients']}")
            print(f"📝 Total Langkah: {rec['total_steps']}")
            print(f"🖼️  Image URL: {rec['image_url']}")
            
            # Tampilkan bahan-bahan (potong jika terlalu panjang)
            ingredients = rec['ingredients_cleaned']
            if len(ingredients) > 200:
                ingredients = ingredients[:200] + "..."
            print(f"\n🥬 Bahan-bahan:\n{ingredients}")
            
            # Tampilkan langkah-langkah (potong jika terlalu panjang)  
            steps = rec['steps_cleaned']
            if len(steps) > 300:
                steps = steps[:300] + "..."
            print(f"\n👨‍🍳 Cara Membuat:\n{steps}")
            
            if i < len(recommendations):
                print("\n" + "="*80)
    
    def run_inference_testing(self, test_user_ids=None, n_test_users=5):
        """Menjalankan inference testing untuk beberapa user"""
        print("\n" + "="*80)
        print("🧪 INFERENCE TESTING - SISTEM REKOMENDASI RESEP INDONESIA")
        print("="*80)
        
        if test_user_ids is None:
            # Pilih random users untuk testing
            all_users = list(self.processed_data['user_id'].unique())
            test_user_ids = np.random.choice(all_users, min(n_test_users, len(all_users)), replace=False)
        
        print(f"🎯 Testing dengan {len(test_user_ids)} users...")
        
        test_results = {}
        
        for i, user_id in enumerate(test_user_ids, 1):
            print(f"\n{'='*20} TEST USER {i}/{len(test_user_ids)} {'='*20}")
            
            try:
                # Test dengan berbagai skenario
                scenarios = [
                    {"name": "Default", "params": {}},
                    {"name": "Mudah saja", "params": {"difficulty_max": 1.5}},
                    {"name": "Rating tinggi", "params": {"min_rating": 4.0}},
                    {"name": "Top 5", "params": {"top_k": 5}}
                ]
                
                user_results = {}
                
                for scenario in scenarios:
                    print(f"\n📋 Skenario: {scenario['name']}")
                    print("-" * 30)
                    
                    recommendations = self.get_enhanced_recommendations(
                        user_id, 
                        show_detailed=False,  # Tidak tampilkan detail untuk testing
                        **scenario['params']
                    )
                    
                    user_results[scenario['name']] = {
                        'count': len(recommendations),
                        'avg_predicted_rating': np.mean([r['predicted_rating'] for r in recommendations]) if recommendations else 0,
                        'categories': list(set([r['category'] for r in recommendations])),
                        'difficulty_levels': list(set([r['difficulty_level'] for r in recommendations]))
                    }
                    
                    print(f"✅ {len(recommendations)} rekomendasi ditemukan")
                    if recommendations:
                        print(f"📊 Avg Predicted Rating: {user_results[scenario['name']]['avg_predicted_rating']:.2f}")
                        print(f"🏷️  Kategori: {', '.join(user_results[scenario['name']]['categories'][:3])}")
                
                test_results[user_id] = user_results
                
            except Exception as e:
                print(f"❌ Error testing user {user_id}: {e}")
                test_results[user_id] = {"error": str(e)}
        
        # Summary hasil testing
        print(f"\n{'='*20} SUMMARY INFERENCE TESTING {'='*20}")
        
        successful_tests = sum(1 for result in test_results.values() if 'error' not in result)
        print(f"✅ Successful tests: {successful_tests}/{len(test_user_ids)}")
        
        if successful_tests > 0:
            # Hitung rata-rata performa
            all_counts = []
            all_ratings = []
            all_categories = set()
            
            for user_result in test_results.values():
                if 'error' not in user_result:
                    default_result = user_result.get('Default', {})
                    if default_result.get('count', 0) > 0:
                        all_counts.append(default_result['count'])
                        all_ratings.append(default_result['avg_predicted_rating'])
                        all_categories.update(default_result['categories'])
            
            if all_counts:
                print(f"📊 Rata-rata jumlah rekomendasi: {np.mean(all_counts):.1f}")
                print(f"⭐ Rata-rata predicted rating: {np.mean(all_ratings):.2f}")
                print(f"🏷️  Total kategori yang direkomendasikan: {len(all_categories)}")
                print(f"🎯 Diversity score: {len(all_categories)/len(self.encoders['category'].classes_):.2f}")
        
        return test_results
    
    def evaluate_model(self, test_data, test_indices):
        """Evaluasi model dengan metrik yang benar"""
        X_test, y_test = test_data
        
        # Basic metrics dari test set yang benar
        predictions = self.model.predict(X_test, verbose=0).flatten()
        rmse = np.sqrt(np.mean((predictions - y_test) ** 2))
        mae = np.mean(np.abs(predictions - y_test))
        
        # Precision@K dan Hit Rate dengan sampling yang lebih realistis
        precision_scores = []
        hit_rates = []
        category_counts = []
        
        # Ambil user dari test set saja untuk evaluasi yang fair
        test_user_ids = X_test['user_id']
        unique_test_users = np.unique(test_user_ids)
        
        # Sample maksimal 50 user untuk evaluasi (lebih realistis)
        n_sample_users = min(50, len(unique_test_users))
        sample_users = np.random.choice(unique_test_users, n_sample_users, replace=False)
        
        valid_evaluations = 0
        for user_encoded in sample_users:
            try:
                # Dapatkan user_id asli
                user_id = self.encoders['user'].inverse_transform([user_encoded])[0]
                
                # Dapatkan rekomendasi
                recs = self.get_enhanced_recommendations(user_id, top_k=10, show_detailed=False)
                
                if recs and len(recs) > 0:
                    # Precision: proporsi rekomendasi dengan predicted rating >= 4.0
                    high_quality_recs = sum(1 for r in recs if r['predicted_rating'] >= 3.5)
                    precision = high_quality_recs / len(recs)
                    precision_scores.append(precision)
                    
                    # Hit rate: apakah ada minimal 1 rekomendasi berkualitas
                    hit_rates.append(1 if high_quality_recs > 0 else 0)
                    
                    # Hitung keragaman kategori untuk user ini
                    user_categories = list(set([r['category'] for r in recs]))
                    category_counts.extend(user_categories)
                    
                    valid_evaluations += 1
                    
            except Exception as e:
                continue
        
        # Hitung metrik akhir dengan penanganan edge case
        avg_precision = np.mean(precision_scores) if precision_scores else 0
        hit_rate = np.mean(hit_rates) if hit_rates else 0
        
        # Diversity: proporsi kategori unik yang muncul dalam rekomendasi
        unique_recommended_categories = len(set(category_counts))
        total_categories = len(self.encoders['category'].classes_)
        diversity = unique_recommended_categories / total_categories if total_categories > 0 else 0
        
        return {
            'rmse': rmse,
            'mae': mae,
            'precision_at_10': avg_precision,
            'hit_rate_at_10': hit_rate,
            'diversity': diversity,
            'n_evaluated_users': valid_evaluations
        }
    
    def save_model(self, filepath):
        """Simpan model dan komponen"""
        # Save model
        self.model.save(f"{filepath}_model.h5")
        
        # Save encoders and other components
        with open(f"{filepath}_components.pkl", 'wb') as f:
            pickle.dump({
                'encoders': self.encoders,
                'scalers': self.scalers,
                'tfidf_vectorizer': self.tfidf_vectorizer,
                'category_similarity': self.category_similarity
            }, f)
        
        print(f"✅ Model tersimpan di {filepath}")
    
    def load_model(self, filepath):
        """Load model dan komponen"""
        try:
            # Load model
            self.model = tf.keras.models.load_model(f"{filepath}_model.h5")
            
            # Load encoders and other components
            with open(f"{filepath}_components.pkl", 'rb') as f:
                components = pickle.load(f)
                self.encoders = components['encoders']
                self.scalers = components['scalers']
                self.tfidf_vectorizer = components['tfidf_vectorizer']
                self.category_similarity = components['category_similarity']
            
            print(f"✅ Model berhasil dimuat dari {filepath}")
            return True
            
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return False

def train_enhanced_indonesian_recipe_model(df, test_size=0.2, validation_size=0.1, epochs=100, batch_size=512):
    """Fungsi utama untuk training enhanced model rekomendasi resep Indonesia"""
    
    print("=" * 60)
    print("🍽️  ENHANCED SISTEM REKOMENDASI RESEP MAKANAN INDONESIA")
    print("=" * 60)
    
    # Initialize enhanced recommender
    recommender = EnhancedIndonesianRecipeRecommender()
    
    # Preprocess data
    processed_data = recommender.preprocess_data(df)
    
    # Train model
    results = recommender.train_model(
        processed_data, 
        test_size=test_size,
        validation_size=validation_size,
        epochs=epochs,
        batch_size=batch_size
    )
    
    # Evaluate model
    print("\n📊 Evaluating model performance...")
    evaluation = recommender.evaluate_model(results['test_data'], results['test_indices'])
    
    # Validation criteria
    validation_criteria = {
        'rmse': 0.25,
        'mae': 0.20,
        'precision_at_10': 0.5,
        'hit_rate_at_10': 0.7,
        'diversity': 0.3
    }
    
    print("\n" + "=" * 60)
    print("📈 HASIL EVALUASI MODEL")
    print("=" * 60)
    print(f"RMSE: {evaluation['rmse']:.4f} (Target: < {validation_criteria['rmse']})")
    print(f"MAE: {evaluation['mae']:.4f} (Target: < {validation_criteria['mae']})")
    print(f"Precision@10: {evaluation['precision_at_10']:.4f} (Target: > {validation_criteria['precision_at_10']})")
    print(f"Hit Rate@10: {evaluation['hit_rate_at_10']:.4f} (Target: > {validation_criteria['hit_rate_at_10']})")
    print(f"Diversity: {evaluation['diversity']:.4f} (Target: > {validation_criteria['diversity']})")
    print(f"Users Evaluated: {evaluation['n_evaluated_users']}")
    
    # Check validation
    validation_passed = all([
        evaluation['rmse'] < validation_criteria['rmse'],
        evaluation['mae'] < validation_criteria['mae'],
        evaluation['precision_at_10'] > validation_criteria['precision_at_10'],
        evaluation['hit_rate_at_10'] > validation_criteria['hit_rate_at_10'],
        evaluation['diversity'] > validation_criteria['diversity']
    ])
    
    print("\n" + "=" * 60)
    print("✅ VALIDASI MODEL" if validation_passed else "❌ VALIDASI MODEL")
    print("=" * 60)
    
    if validation_passed:
        print("🎉 Model siap untuk production!")
        print("Model memenuhi semua kriteria kualitas untuk rekomendasi resep Indonesia.")
    else:
        print("⚠️  Model perlu perbaikan sebelum deployment.")
    
    return {
        'recommender': recommender,
        'model': recommender.model,
        'processed_data': processed_data,
        'evaluation': evaluation,
        'validation_passed': validation_passed,
        'training_history': results['history']
    }


In [6]:
# =============================================================================
# EXAMPLE USAGE DAN INFERENCE TESTING
# =============================================================================

# 1. TRAINING MODEL
df = pd.read_csv('../data/data_recipes_cleaned.csv')
results = train_enhanced_indonesian_recipe_model(df)
recommender = results['recommender']

# 2. SAVE MODEL
recommender.save_model('indonesian_recipe_model')

# # 3. LOAD MODEL (untuk production)
new_recommender = EnhancedIndonesianRecipeRecommender()
# new_recommender.load_model('indonesian_recipe_model')
# new_recommender.processed_data = processed_data  # Load processed data
# new_recommender.original_data = original_data    # Load original data
    
#     # 4. INFERENCE TESTING
# test_results = recommender.run_inference_testing(n_test_users=10)


  

🍽️  ENHANCED SISTEM REKOMENDASI RESEP MAKANAN INDONESIA
🔄 Memulai preprocessing data...
📝 Preprocessing teks bahasa Indonesia...
⚙️ Menghitung tingkat kesulitan resep...
🏷️ Encoding kategori...
📊 Membuat TF-IDF vectors...
✅ Preprocessing selesai!
🚀 Memulai training model...
🏗️ Membangun model rekomendasi hybrid...

Epoch 1/100


21/21 [==============================] - 4s 36ms/step - loss: 0.0729 - mae: 0.2142 - val_loss: 0.0721 - val_mae: 0.2270 - lr: 0.0010
Epoch 2/100
21/21 [==============================] - 0s 20ms/step - loss: 0.0563 - mae: 0.1896 - val_loss: 0.0638 - val_mae: 0.2083 - lr: 0.0010
Epoch 3/100
21/21 [==============================] - 0s 20ms/step - loss: 0.0517 - mae: 0.1820 - val_loss: 0.0570 - val_mae: 0.1996 - lr: 0.0010
Epoch 4/100
21/21 [==============================] - 0s 22ms/step - loss: 0.0480 - mae: 0.1739 - val_loss: 0.0535 - val_mae: 0.1952 - lr: 0.0010
Epoch 5/100
21/21 [==============================] - 1s 24ms/step - loss: 0.0439 - mae: 0.1668 - val_

In [3]:
# # 5. GET RECOMMENDATIONS UNTUK USER TERTENTU
# user_id = "user_123"
# recommendations = recommender.get_enhanced_recommendations(
#     user_id=user_id,
#     top_k=10,
#     category_filter=None,      # Filter kategori tertentu
#     difficulty_max=3,          # Maksimal tingkat kesulitan
#     min_rating=3.0,           # Rating minimum
#     show_detailed=True        # Tampilkan output lengkap
# )

In [4]:

    
#     # 6. SKENARIO TESTING BERBEDA
# test_scenarios = [
#     # User existing dengan berbagai filter
#     {"user_id": "user_123", "category_filter": "Makanan Utama"},
#     {"user_id": "user_456", "difficulty_max": 2, "min_rating": 4.0},
    
#     # User baru (content-based)
#     {"user_id": "new_user_999", "top_k": 5},
# ]

# for scenario in test_scenarios:
#     recs = recommender.get_enhanced_recommendations(**scenario)
#     print(f"User {scenario['user_id']}: {len(recs)} recommendations")

In [ ]:
# """
# SCRIPT INFERENCE TESTING UNTUK INDONESIAN RECIPE RECOMMENDER
# ============================================================

# Script ini untuk menjalankan testing dan mendapatkan rekomendasi dengan output lengkap
# seperti Netflix untuk sistem rekomendasi resep Indonesia Anda.

# Usage:
# 1. Jalankan training terlebih dahulu
# 2. Gunakan script ini untuk testing dan inference
# """

# import pandas as pd
# import numpy as np

# class RecipeRecommenderTester:
#     """Class untuk testing sistem rekomendasi resep"""
    
#     def __init__(self, recommender=None):
#         self.recommender = recommender
        
#     def full_pipeline_test(self, df_path_or_dataframe, model_save_path="indonesian_recipe_model"):
#         """Menjalankan full pipeline dari training hingga testing"""
        
#         print("🚀 FULL PIPELINE TESTING")
#         print("=" * 60)
        
#         # 1. Load atau gunakan dataframe
#         if isinstance(df_path_or_dataframe, str):
#             print(f"📂 Loading data dari {df_path_or_dataframe}")
#             df = pd.read_csv(df_path_or_dataframe)
#         else:
#             df = df_path_or_dataframe.copy()
            
#         print(f"📊 Data shape: {df.shape}")
        
#         # 2. Training model
#         print("\n🎯 STEP 1: TRAINING MODEL")
#         print("-" * 40)
        
#         results = train_enhanced_indonesian_recipe_model(
#             df, 
#             epochs=50,  # Kurangi epochs untuk testing
#             batch_size=256
#         )
        
#         self.recommender = results['recommender']
        
#         # 3. Save model
#         print(f"\n💾 STEP 2: SAVING MODEL ke {model_save_path}")
#         print("-" * 40)
#         self.recommender.save_model(model_save_path)
        
#         # 4. Inference testing
#         print("\n🧪 STEP 3: INFERENCE TESTING")
#         print("-" * 40)
#         test_results = self.recommender.run_inference_testing(n_test_users=8)
        
#         # 5. Detailed testing untuk beberapa user
#         print("\n🎯 STEP 4: DETAILED TESTING")
#         print("-" * 40)
#         self.detailed_user_testing()
        
#         return {
#             'training_results': results,
#             'test_results': test_results,
#             'recommender': self.recommender
#         }
    
#     def detailed_user_testing(self):
#         """Testing detail untuk beberapa user dengan berbagai skenario"""
        
#         if self.recommender is None:
#             print("❌ Recommender belum diinisialisasi!")
#             return
        
#         # Ambil sample user IDs
#         sample_users = list(self.recommender.processed_data['user_id'].unique())[:5]
        
#         test_scenarios = [
#             {
#                 "name": "🎯 Rekomendasi Default",
#                 "params": {"top_k": 8, "show_detailed": True}
#             },
#             {
#                 "name": "😊 Resep Mudah Saja", 
#                 "params": {"difficulty_max": 1.5, "top_k": 5, "show_detailed": False}
#             },
#             {
#                 "name": "⭐ Rating Tinggi (4.0+)",
#                 "params": {"min_rating": 4.0, "top_k": 5, "show_detailed": False}
#             },
#             {
#                 "name": "🥘 Makanan Utama Saja",
#                 "params": {"category_filter": "Makanan Utama", "top_k": 3, "show_detailed": False}
#             }
#         ]
        
#         for i, user_id in enumerate(sample_users[:3], 1):  # Test 3 users saja
#             print(f"\n{'='*20} TESTING USER {i}: {user_id} {'='*20}")
            
#             for scenario in test_scenarios:
#                 print(f"\n{scenario['name']}")
#                 print("-" * 50)
                
#                 try:
#                     recommendations = self.recommender.get_enhanced_recommendations(
#                         user_id=user_id,
#                         **scenario['params']
#                     )
                    
#                     if not scenario['params'].get('show_detailed', False):
#                         # Tampilkan summary
#                         print(f"✅ {len(recommendations)} rekomendasi ditemukan")
#                         if recommendations:
#                             avg_rating = np.mean([r['predicted_rating'] for r in recommendations])
#                             categories = list(set([r['category'] for r in recommendations]))
#                             difficulties = list(set([r['difficulty_level'] for r in recommendations]))
                            
#                             print(f"📊 Avg Rating: {avg_rating:.2f}")
#                             print(f"🏷️  Kategori: {', '.join(categories[:3])}")
#                             print(f"📈 Tingkat Kesulitan: {', '.join(difficulties)}")
                            
#                             # Tampilkan top 3 judul
#                             top_titles = [r['title_cleaned'][:50] + "..." if len(r['title_cleaned']) > 50 
#                                         else r['title_cleaned'] for r in recommendations[:3]]
#                             print(f"🥘 Top Recipes: {', '.join(top_titles)}")
                        
#                 except Exception as e:
#                     print(f"❌ Error: {e}")
    
#     def test_new_user_scenario(self):
#         """Testing skenario untuk user baru (content-based)"""
        
#         print("\n" + "="*60)
#         print("👤 TESTING SKENARIO USER BARU (CONTENT-BASED)")
#         print("="*60)
        
#         # Test dengan user ID yang tidak ada
#         new_user_scenarios = [
#             {
#                 "user_id": "new_user_001",
#                 "name": "🆕 User Baru - Default",
#                 "params": {"top_k": 10}
#             },
#             {
#                 "user_id": "new_user_002", 
#                 "name": "🆕 User Baru - Hanya Resep Mudah",
#                 "params": {"difficulty_max": 1.5, "top_k": 5}
#             },
#             {
#                 "user_id": "new_user_003",
#                 "name": "🆕 User Baru - Rating Tinggi",
#                 "params": {"min_rating": 4.0, "top_k": 5}
#             }
#         ]
        
#         for scenario in new_user_scenarios:
#             print(f"\n{scenario['name']}")
#             print("-" * 50)
            
#             try:
#                 recommendations = self.recommender.get_enhanced_recommendations(
#                     user_id=scenario['user_id'],
#                     show_detailed=True,  # Show detailed untuk user baru
#                     **scenario['params']
#                 )
                
#                 print(f"✅ {len(recommendations)} rekomendasi content-based berhasil dibuat")
                
#             except Exception as e:
#                 print(f"❌ Error: {e}")
    
#     def compare_recommendation_methods(self, user_id=None):
#         """Membandingkan hasil rekomendasi dengan parameter berbeda"""
        
#         if user_id is None:
#             user_id = list(self.recommender.processed_data['user_id'].unique())[0]
        
#         print(f"\n{'='*60}")
#         print(f"🔄 COMPARISON TESTING UNTUK USER: {user_id}")
#         print("="*60)
        
#         comparison_scenarios = [
#             {"name": "Top 5 Default", "params": {"top_k": 5}},
#             {"name": "Top 10 Default", "params": {"top_k": 10}},
#             {"name": "Mudah + Rating Tinggi", "params": {"difficulty_max": 2, "min_rating": 4.0, "top_k": 5}},
#             {"name": "Sulit + Semua Rating", "params": {"difficulty_max": 3, "min_rating": 1.0, "top_k": 5}},
#         ]
        
#         results_comparison = {}
        
#         for scenario in comparison_scenarios:
#             print(f"\n📊 {scenario['name']}")
#             print("-" * 30)
            
#             try:
#                 recommendations = self.recommender.get_enhanced_recommendations(
#                     user_id=user_id,
#                     show_detailed=False,
#                     **scenario['params']
#                 )
                
#                 if recommendations:
#                     avg_rating = np.mean([r['predicted_rating'] for r in recommendations])
#                     categories = len(set([r['category'] for r in recommendations]))
#                     difficulties = list(set([r['difficulty_level'] for r in recommendations]))
                    
#                     results_comparison[scenario['name']] = {
#                         'count': len(recommendations),
#                         'avg_rating': avg_rating,
#                         'categories': categories,
#                         'difficulties': difficulties
#                     }
                    
#                     print(f"  📈 Count: {len(recommendations)}")
#                     print(f"  ⭐ Avg Rating: {avg_rating:.2f}")
#                     print(f"  🏷️  Categories: {categories}")
#                     print(f"  📊 Difficulties: {', '.join(difficulties)}")
#                 else:
#                     print("  ❌ Tidak ada rekomendasi")
#                     results_comparison[scenario['name']] = None
                    
#             except Exception as e:
#                 print(f"  ❌ Error: {e}")
#                 results_comparison[scenario['name']] = None
        
#         # Summary comparison
#         print(f"\n📋 SUMMARY COMPARISON")
#         print("-" * 40)
#         for name, result in results_comparison.items():
#             if result:
#                 print(f"{name}: {result['count']} recs, {result['avg_rating']:.2f} avg rating")
#             else:
#                 print(f"{name}: No recommendations")
        
#         return results_comparison
    
#     def load_and_test_saved_model(self, model_path, processed_data, original_data):
#         """Load model yang sudah disave dan test"""
        
#         print("\n" + "="*60)
#         print("💾 TESTING SAVED MODEL")
#         print("="*60)
        
#         # Initialize new recommender
#         new_recommender = EnhancedIndonesianRecipeRecommender()
        
#         # Load model
#         success = new_recommender.load_model(model_path)
        
#         if not success:
#             print("❌ Gagal load model!")
#             return None
        
#         # Set data
#         new_recommender.processed_data = processed_data
#         new_recommender.original_data = original_data
        
#         # Test loaded model
#         print("\n🧪 Testing loaded model...")
#         sample_user = list(processed_data['user_id'].unique())[0]
        
#         recommendations = new_recommender.get_enhanced_recommendations(
#             user_id=sample_user,
#             top_k=5,
#             show_detailed=True
#         )
        
#         print(f"✅ Loaded model berhasil memberikaan {len(recommendations)} rekomendasi!")
        
#         return new_recommender



In [ ]:
#     # Load your data
# df = pd.read_csv('../data/data_recipes_cleaned.csv')

# # Initialize tester
# tester = RecipeRecommenderTester()

# # Run full pipeline
# results = tester.full_pipeline_test(df)


🚀 FULL PIPELINE TESTING
📊 Data shape: (14635, 12)

🎯 STEP 1: TRAINING MODEL
----------------------------------------
🍽️  ENHANCED SISTEM REKOMENDASI RESEP MAKANAN INDONESIA
🔄 Memulai preprocessing data...
📝 Preprocessing teks bahasa Indonesia...
⚙️ Menghitung tingkat kesulitan resep...
🏷️ Encoding kategori...
📊 Membuat TF-IDF vectors...
✅ Preprocessing selesai!
🚀 Memulai training model...
🏗️ Membangun model rekomendasi hybrid...
Epoch 1/50
41/41 [==============================] - 4s 23ms/step - loss: 0.0612 - mae: 0.1985 - val_loss: 0.0612 - val_mae: 0.2038 - lr: 0.0010
Epoch 2/50
41/41 [==============================] - 1s 16ms/step - loss: 0.0503 - mae: 0.1811 - val_loss: 0.0472 - val_mae: 0.1862 - lr: 0.0010
Epoch 3/50
41/41 [==============================] - 1s 15ms/step - loss: 0.0464 - mae: 0.1734 - val_loss: 0.0454 - val_mae: 0.1821 - lr: 0.0010
Epoch 4/50
41/41 [==============================] - 1s 16ms/step - loss: 0.0410 - mae: 0.1618 - val_loss: 0.0437 - val_mae: 0.1706 - lr

KeyboardInterrupt: 

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import pickle

class UserPreferenceHandler:
    """Handler untuk menangani preferensi user baru berdasarkan form sign-up"""
    
    def __init__(self):
        self.category_profiles = {}
        self.difficulty_profiles = {}
        self.ingredient_profiles = {}
        
    def build_category_profiles(self, data):
        """Membangun profil karakteristik untuk setiap kategori"""
        print("📊 Membangun profil kategori...")
        
        category_profiles = {}
        
        for category in data['Category'].unique():
            category_data = data[data['Category'] == category]
            
            profile = {
                'avg_rating': category_data['rating'].mean(),
                'avg_total_rating': category_data['total_rating'].mean(),
                'avg_ingredients': category_data['Total Ingredients'].mean(),
                'avg_steps': category_data['Total Steps'].mean(),
                'difficulty_distribution': {
                    'mudah': len(category_data[category_data['Total Ingredients'] <= 5]) / len(category_data),
                    'sedang': len(category_data[(category_data['Total Ingredients'] > 5) & 
                                              (category_data['Total Ingredients'] <= 10)]) / len(category_data),
                    'sulit': len(category_data[category_data['Total Ingredients'] > 10]) / len(category_data)
                },
                'popular_ingredients': self._extract_popular_ingredients(category_data),
                'cooking_methods': self._extract_cooking_methods(category_data),
                'total_recipes': len(category_data)
            }
            
            category_profiles[category] = profile
        
        self.category_profiles = category_profiles
        return category_profiles
    
    def _extract_popular_ingredients(self, category_data, top_n=10):
        """Ekstrak bahan-bahan populer dalam kategori"""
        all_ingredients = []
        for ingredients in category_data['Ingredients Cleaned'].dropna():
            # Split dan clean ingredients
            ing_list = [ing.strip().lower() for ing in str(ingredients).split(',')]
            all_ingredients.extend(ing_list)
        
        ingredient_freq = Counter(all_ingredients)
        return [ing for ing, freq in ingredient_freq.most_common(top_n)]
    
    def _extract_cooking_methods(self, category_data, top_n=5):
        """Ekstrak metode memasak populer dalam kategori"""
        cooking_methods = ['goreng', 'rebus', 'kukus', 'bakar', 'panggang', 'tumis', 'sangrai']
        method_counts = {}
        
        for method in cooking_methods:
            count = 0
            for steps in category_data['Steps Cleaned'].dropna():
                if method in str(steps).lower():
                    count += 1
            method_counts[method] = count / len(category_data)
        
        return sorted(method_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]

class EnhancedColdStartRecommender:
    """Enhanced recommender system dengan cold-start handling yang lebih baik"""
    
    def __init__(self, base_recommender):
        self.base_recommender = base_recommender
        self.preference_handler = UserPreferenceHandler()
        
    def setup_cold_start_profiles(self, data):
        """Setup profil untuk cold-start recommendations"""
        self.preference_handler.build_category_profiles(data)
        print("✅ Cold-start profiles berhasil dibuat!")
    
    def get_new_user_recommendations_with_preferences(self, user_preferences, top_k=10):
        """
        Mendapatkan rekomendasi untuk user baru berdasarkan preferensi
        
        Args:
            user_preferences (dict): {
                'preferred_categories': ['Masakan Nusantara', 'Cemilan'],
                'difficulty_preference': 'mudah', # 'mudah', 'sedang', 'sulit', 'semua'
                'max_ingredients': 10,
                'max_steps': 8,
                'cooking_methods': ['goreng', 'rebus'], # optional
                'dietary_restrictions': [], # optional untuk masa depan
                'spice_level': 'sedang' # optional untuk masa depan
            }
        """
        
        print(f"\n🆕 Memberikan rekomendasi untuk USER BARU")
        print("=" * 60)
        print(f"✅ Kategori preferensi: {user_preferences.get('preferred_categories', [])}")
        print(f"✅ Tingkat kesulitan: {user_preferences.get('difficulty_preference', 'semua')}")
        print(f"✅ Maksimal bahan: {user_preferences.get('max_ingredients', 'tidak dibatasi')}")
        
        data = self.base_recommender.processed_data.copy()
        original_data = self.base_recommender.original_data.copy()
        
        # 1. Filter berdasarkan kategori preferensi
        preferred_categories = user_preferences.get('preferred_categories', [])
        if preferred_categories:
            data = data[data['Category'].isin(preferred_categories)]
            print(f"📊 Resep setelah filter kategori: {len(data)}")
        
        # 2. Filter berdasarkan tingkat kesulitan
        difficulty_pref = user_preferences.get('difficulty_preference', 'semua')
        if difficulty_pref != 'semua':
            if difficulty_pref == 'mudah':
                data = data[data['Total Ingredients'] <= 7]
                data = data[data['Total Steps'] <= 6]
            elif difficulty_pref == 'sedang':
                data = data[(data['Total Ingredients'] > 7) & (data['Total Ingredients'] <= 12)]
                data = data[(data['Total Steps'] > 6) & (data['Total Steps'] <= 10)]
            elif difficulty_pref == 'sulit':
                data = data[data['Total Ingredients'] > 12]
                data = data[data['Total Steps'] > 10]
            print(f"📊 Resep setelah filter kesulitan: {len(data)}")
        
        # 3. Filter berdasarkan maksimal bahan dan langkah
        max_ingredients = user_preferences.get('max_ingredients')
        max_steps = user_preferences.get('max_steps')
        
        if max_ingredients:
            data = data[data['Total Ingredients'] <= max_ingredients]
            print(f"📊 Resep setelah filter max bahan: {len(data)}")
            
        if max_steps:
            data = data[data['Total Steps'] <= max_steps]
            print(f"📊 Resep setelah filter max langkah: {len(data)}")
        
        # 4. Filter berdasarkan metode memasak (jika ada)
        cooking_methods = user_preferences.get('cooking_methods', [])
        if cooking_methods:
            filtered_items = []
            for _, row in data.iterrows():
                steps_text = str(row['Steps Cleaned']).lower()
                if any(method in steps_text for method in cooking_methods):
                    filtered_items.append(row['item_id'])
            
            if filtered_items:
                data = data[data['item_id'].isin(filtered_items)]
                print(f"📊 Resep setelah filter metode masak: {len(data)}")
        
        if len(data) == 0:
            print("❌ Tidak ada resep yang memenuhi kriteria preferensi")
            return []
        
        # 5. Scoring berdasarkan preferensi
        data = self._calculate_preference_scores(data, user_preferences)
        
        # 6. Ambil data asli untuk output
        item_ids = data['item_id'].tolist()
        original_filtered = original_data[original_data['item_id'].isin(item_ids)]
        
        # 7. Merge dan sort berdasarkan preference score
        merged_data = data.merge(
            original_filtered[['item_id', 'Title Cleaned', 'Steps Cleaned', 'Ingredients Cleaned', 'Image URL']], 
            on='item_id'
        )
        
        merged_data = merged_data.sort_values(['preference_score', 'total_rating'], ascending=False)
        
        # 8. Buat recommendations
        recommendations = []
        for _, row in merged_data.head(top_k).iterrows():
            # Hitung difficulty level
            if row['Total Ingredients'] <= 7 and row['Total Steps'] <= 6:
                difficulty_level = "Mudah"
            elif row['Total Ingredients'] <= 12 and row['Total Steps'] <= 10:
                difficulty_level = "Sedang"
            else:
                difficulty_level = "Sulit"
            
            # Hitung match score dengan preferensi user
            match_score = self._calculate_match_score(row, user_preferences)
            
            recommendations.append({
                'item_id': row['item_id'],
                'title_cleaned': row['Title Cleaned'],
                'steps_cleaned': row['Steps Cleaned'],
                'ingredients_cleaned': row['Ingredients Cleaned'],
                'category': row['Category'],
                'total_rating': row['total_rating'],
                'image_url': row.get('Image URL', 'N/A'),
                'predicted_rating': row['total_rating'],  # Untuk user baru
                'difficulty_level': difficulty_level,
                'total_ingredients': row['Total Ingredients'],
                'total_steps': row['Total Steps'],
                'preference_score': row['preference_score'],
                'match_score': match_score,
                'user_type': 'new_with_preferences',
                'recommendation_reason': self._generate_recommendation_reason(row, user_preferences)
            })
        
        # Display recommendations
        self._display_preference_based_recommendations(recommendations, user_preferences)
        
        return recommendations
    
    def _calculate_preference_scores(self, data, user_preferences):
        """Menghitung skor preferensi untuk setiap resep"""
        data = data.copy()
        scores = []
        
        for _, row in data.iterrows():
            score = 0
            
            # Base score dari popularitas
            score += (row['total_rating'] / data['total_rating'].max()) * 0.3
            score += (row['rating'] / 5.0) * 0.2
            
            # Kategori bonus
            if row['Category'] in user_preferences.get('preferred_categories', []):
                score += 0.3
            
            # Difficulty preference bonus
            difficulty_pref = user_preferences.get('difficulty_preference', 'semua')
            total_ingredients = row['Total Ingredients']
            total_steps = row['Total Steps']
            
            if difficulty_pref == 'mudah' and total_ingredients <= 7 and total_steps <= 6:
                score += 0.2
            elif difficulty_pref == 'sedang' and 7 < total_ingredients <= 12 and 6 < total_steps <= 10:
                score += 0.2
            elif difficulty_pref == 'sulit' and total_ingredients > 12 and total_steps > 10:
                score += 0.2
            elif difficulty_pref == 'semua':
                score += 0.1
            
            scores.append(score)
        
        data['preference_score'] = scores
        return data
    
    def _calculate_match_score(self, row, user_preferences):
        """Menghitung seberapa cocok resep dengan preferensi user"""
        match_score = 0
        total_criteria = 0
        
        # Kategori match
        if user_preferences.get('preferred_categories'):
            total_criteria += 1
            if row['Category'] in user_preferences['preferred_categories']:
                match_score += 1
        
        # Difficulty match
        difficulty_pref = user_preferences.get('difficulty_preference', 'semua')
        if difficulty_pref != 'semua':
            total_criteria += 1
            if difficulty_pref == 'mudah' and row['Total Ingredients'] <= 7:
                match_score += 1
            elif difficulty_pref == 'sedang' and 7 < row['Total Ingredients'] <= 12:
                match_score += 1
            elif difficulty_pref == 'sulit' and row['Total Ingredients'] > 12:
                match_score += 1
        
        # Ingredients limit match
        if user_preferences.get('max_ingredients'):
            total_criteria += 1
            if row['Total Ingredients'] <= user_preferences['max_ingredients']:
                match_score += 1
        
        # Steps limit match
        if user_preferences.get('max_steps'):
            total_criteria += 1
            if row['Total Steps'] <= user_preferences['max_steps']:
                match_score += 1
        
        return (match_score / total_criteria * 100) if total_criteria > 0 else 0
    
    def _generate_recommendation_reason(self, row, user_preferences):
        """Generate alasan mengapa resep ini direkomendasikan"""
        reasons = []
        
        if row['Category'] in user_preferences.get('preferred_categories', []):
            reasons.append(f"Sesuai kategori favorit ({row['Category']})")
        
        if row['total_rating'] >= 4.0:
            reasons.append("Rating tinggi dari pengguna lain")
        
        difficulty_pref = user_preferences.get('difficulty_preference', 'semua')
        if difficulty_pref == 'mudah' and row['Total Ingredients'] <= 7:
            reasons.append("Mudah dibuat dengan bahan sedikit")
        elif difficulty_pref == 'sedang':
            reasons.append("Tingkat kesulitan sedang")
        
        if row['Total Steps'] <= 5:
            reasons.append("Langkah memasak singkat")
        
        return "; ".join(reasons) if reasons else "Resep populer dan berkualitas"
    
    def _display_preference_based_recommendations(self, recommendations, user_preferences):
        """Display recommendations dengan informasi preferensi"""
        print(f"\n🍽️  REKOMENDASI BERDASARKAN PREFERENSI USER BARU")
        print("=" * 80)
        
        if not recommendations:
            print("❌ Tidak ada rekomendasi yang memenuhi preferensi")
            return
        
        print(f"📊 Total rekomendasi: {len(recommendations)}")
        print(f"🎯 Kategori preferensi: {', '.join(user_preferences.get('preferred_categories', []))}")
        print("=" * 80)
        
        for i, rec in enumerate(recommendations, 1):
            print(f"\n🥘 REKOMENDASI #{i}")
            print("-" * 50)
            print(f"📍 Item ID: {rec['item_id']}")
            print(f"🍲 Judul: {rec['title_cleaned']}")
            print(f"🏷️  Kategori: {rec['category']}")
            print(f"⭐ Rating: {rec['total_rating']:.1f}")
            print(f"📊 Tingkat Kesulitan: {rec['difficulty_level']}")
            print(f"🧾 Bahan: {rec['total_ingredients']} | Langkah: {rec['total_steps']}")
            print(f"🎯 Match Score: {rec['match_score']:.1f}%")
            print(f"💡 Alasan: {rec['recommendation_reason']}")
            
            if i < len(recommendations):
                print("\n" + "-"*50)

# Integrasi dengan sistem utama
def integrate_with_main_system(base_recommender_class):
    """Mengintegrasikan cold-start handler dengan sistem utama"""
    
    # Modifikasi method get_enhanced_recommendations
    original_get_recommendations = base_recommender_class.get_enhanced_recommendations
    
    def enhanced_get_recommendations_with_preferences(self, user_id, user_preferences=None, **kwargs):
        """Enhanced version yang mendukung user preferences"""
        
        # Cek apakah user baru
        try:
            user_encoded = self.encoders['user'].transform([user_id])[0]
            user_type = "existing"
        except:
            user_type = "new"
            
            # Jika user baru dan ada preferensi, gunakan preference-based recommendations
            if user_preferences:
                if not hasattr(self, 'cold_start_recommender'):
                    self.cold_start_recommender = EnhancedColdStartRecommender(self)
                    self.cold_start_recommender.setup_cold_start_profiles(self.original_data)
                
                return self.cold_start_recommender.get_new_user_recommendations_with_preferences(
                    user_preferences, kwargs.get('top_k', 10)
                )
            else:
                # Fallback ke content-based biasa
                return self._get_enhanced_content_based_recommendations(
                    kwargs.get('category_filter'),
                    kwargs.get('difficulty_max', 3),
                    kwargs.get('top_k', 10),
                    kwargs.get('min_rating', 3.0),
                    kwargs.get('show_detailed', True)
                )
        
        # User existing, gunakan collaborative filtering biasa
        return original_get_recommendations(user_id, **kwargs)
    
    # Replace method
    base_recommender_class.get_enhanced_recommendations_with_preferences = enhanced_get_recommendations_with_preferences
    
    return base_recommender_class

In [5]:
new_user_preferences = {
        'preferred_categories': ['ayam', 'tempe', 'daging'],
        'difficulty_preference': 'mudah',  # mudah, sedang, sulit, atau semua
        'max_ingredients': 8,
        'max_steps': 6,
        'cooking_methods': ['goreng', 'rebus'],  # optional
}
    
    # Contoh penggunaan (setelah model di-train)
recommendations = new_recommender.get_enhanced_recommendations_with_preferences(
        user_id="NEW_USER_123",
        user_preferences=new_user_preferences,
        top_k=10
)

AttributeError: 'EnhancedIndonesianRecipeRecommender' object has no attribute 'get_enhanced_recommendations_with_preferences'